# Rajput EL-to-IV Pipeline — Joji's working copy

Adapted from Zubair's `rajput_implement.py`. Edits made for Joji's machine
are marked with `# EDIT:` comments — search for that string to find every
place you need to check or fill in before running.

**Before running, do these in order:**
1. Fix `source_dir` (Step 0 cell below) to point at your dataset folder.
2. Either get `Pickle.py` / `Matplotlib_stylesheet.py` from Zubair/supervisor
   and drop them in a `support_scripts` folder, OR leave the fallback below
   active (it skips those imports safely if the files aren't found).
3. Run the "bit-depth check" cell early — confirms whether your TIFFs are
   16-bit, which decides whether the image-loading fix (Step 4 below) is
   needed.
4. Ngspice is already wired up for your `C:\ngspice_dll` install — no edit
   needed there unless you move that folder.

In [ ]:
'''
Rajput EL-to-IV parameter extraction — refactored into functions.
Reference: Rajput et al., Sol. Energy 2018, DOI: 10.1016/j.solener.2018.07.046
Applied to the Sandia PV-IV-EL dataset.

Each pipeline step has a corresponding validate_* function that reproduces
the inline diagnostic plots/prints from rajput_implement.py.

Original: Zubair Abdullah-Vetter
Edited for local run: Joji (Ghozy Abror)
'''

import os
import sys
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.constants import Boltzmann as k_B
from scipy.constants import elementary_charge as q_e
from scipy.constants import zero_Celsius as T_0
from scipy.interpolate import interp1d
from scipy.optimize import least_squares
from skimage import morphology

from pvlib.ivtools.sde import fit_sandia_simple
from pvlib.pvsystem import i_from_v

# directories
cwd = os.getcwd()

# EDIT: this pulled in two helper modules (Pickle.py, Matplotlib_stylesheet.py)
# from Zubair's machine. Neither is called anywhere else in this script as
# far as I can tell, so this is wrapped so a missing folder won't crash the
# whole notebook. If you get the real files from your supervisor, put them
# in a `support_scripts` folder at the path below and this will pick them up
# automatically. If some later cell errors with a NameError for something
# these files defined, that's the sign you actually need them.
libs_path = os.path.abspath(os.path.join(cwd, '..', '..', 'support_scripts'))
sys.path.append(libs_path)
try:
    from Pickle import *
    from Matplotlib_stylesheet import *
    print(f"Loaded helper modules from: {libs_path}")
except ImportError:
    print(f"Note: Pickle.py / Matplotlib_stylesheet.py not found at {libs_path} "
          f"— skipping (not required unless a later cell errors asking for them).")

# EDIT: point this at YOUR dataset root, not Zubair's OneDrive path.
# Fill in the actual folder name(s) below — this is currently a guess based
# on your working-directory convention from other chats. Update as needed.
source_dir = r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding"  # EDIT THIS PATH
cell_dir   = os.path.join(source_dir, "EL_cell_low_res_2")

df = pd.read_excel(os.path.join(source_dir, "AnonDB.xlsx"),
                   sheet_name="AnonDB", usecols="A:BO", header=0, nrows=617)
df["IVPath"]  = df["IVPath"].str.replace("./IV/", "")
df["ELLPath"] = df["ELLPath"].str.replace("./EL/", "")
df["ELHPath"] = df["ELHPath"].str.replace("./EL/", "")

V_th = k_B * (25 + T_0) / q_e  # thermal voltage at 25 degC

In [ ]:
# This must run before any PySpice/ngspice simulation call (i.e. before
# sim_module_oneD() is ever used). Without it you'll hit the same
# "NGSPICE_PATH TypeError" / "cannot load library" errors we debugged earlier.
#
# If you ever move the C:\ngspice_dll folder, update the two paths below to
# match. LIBRARY_PATH must keep the literal `{}` right before `.dll` —
# PySpice fills that placeholder in itself, don't replace it.

try:
    from PySpice.Spice.NgSpice.Shared import NgSpiceShared
    NgSpiceShared.LIBRARY_PATH = r"C:\ngspice_dll\Spice64_dll\dll-vs\ngspice{}.dll"
    NgSpiceShared.NGSPICE_PATH = r"C:\ngspice_dll\Spice64_dll"
    _ng = NgSpiceShared.new_instance()
    print("ngspice loaded OK:", _ng.exec_command('version').splitlines()[0]
          if _ng.exec_command('version') else "(no version string returned)")
except Exception as e:
    print(f"WARNING: ngspice did not load cleanly ({e}). "
          f"PySpice simulation cells later in this notebook will fail until "
          f"this is fixed.")

## Bit-depth check  (EDIT: new cell, not in original script)
Run this once on a real TIFF from your dataset. If `dtype` prints
`uint16` and `max` is well above 255, your source images are genuinely
16-bit — which means the image loading in `load_module_data` below
(currently using `cv2.IMREAD_GRAYSCALE`, an 8-bit read) is discarding
real precision before the ×257 rescale ever runs. See the EDIT note in
that function for the fix, and flag this to your supervisor before
changing it, since it's a deviation from Zubair's original code.

In [ ]:
# EDIT: point this at any single real TIFF from your dataset to check bit depth
_sample_tiff_path = None  # e.g. r"C:\...\EL_cell_low_res\<module>_80\<module>_80_001.tiff"
if _sample_tiff_path:
    _sample = cv2.imread(_sample_tiff_path, cv2.IMREAD_UNCHANGED)
    print("dtype:", _sample.dtype, "| max value:", _sample.max())
    if _sample.dtype == np.uint8:
        print("-> Source is genuinely 8-bit. Current IMREAD_GRAYSCALE loading is fine.")
    else:
        print("-> Source is >8-bit. Current IMREAD_GRAYSCALE loading is DOWNCONVERTING "
              "before the rescale — see EDIT note in load_module_data().")
else:
    print("Set _sample_tiff_path above to run this check.")

In [ ]:

def quick_plot(img, title=None, cmap="inferno", cbar=False):
    """Display a single image with optional colourbar."""
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.xticks([]); plt.yticks([])
    if cbar:
        plt.colorbar(shrink=0.5)
    plt.show()


def plot_module_images(module_imgs, nrows=10, ncols=6, cmap="inferno",
                       cbar=False, title=None, cbar_title=None, save_path=None):
    """Plot a nrows x ncols grid of cell images with shared colour scale."""
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2),
                             gridspec_kw=dict(wspace=0.03, hspace=0.03))
    vmin = np.nanpercentile(module_imgs, 1)
    vmax = np.nanpercentile(module_imgs, 99)
    for idx, ax in enumerate(axes.flat):
        ax.imshow(module_imgs[idx], cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axis("off")
    if cbar:
        cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        fig.colorbar(sm, cax=cbar_ax)
        cbar_ax.yaxis.set_tick_params(labelsize=30)
        cbar_ax.yaxis.set_label_position('right')
        cbar_ax.yaxis.set_label_text(cbar_title, fontsize=40)
    if title:
        plt.suptitle(title, fontsize=16)
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

In [ ]:

def _center_square_mask(h, w, ratio):
    """Return a filled square mask centred on (H, W); side = ratio * min(h, w)."""
    mask = np.zeros((h, w), dtype=np.uint8)
    side = int(min(h, w) * ratio)
    tl = (w // 2 - side // 2, h // 2 - side // 2)
    br = (w // 2 + side // 2, h // 2 + side // 2)
    cv2.rectangle(mask, tl, br, color=1, thickness=-1)
    return mask.astype(np.float32)


def create_busbar_mask(cell, IBC=False, busbar_orientation="horizontal"):
    """
    Build a boolean active-area mask for a cell image.

    - For standard cells (IBC=False): removes busbars detected as intensity
      troughs in the averaged line profile, plus dark-edge pixels.
    - For IBC cells (IBC=True): uses only the percentile and centre-fill masks
      (no busbars to suppress).

    Returns bool array (True = active silicon).
    """
    H, W = cell.shape
    dim = H if busbar_orientation == "horizontal" else W

    # build average line profile perpendicular to busbar direction
    profiles = [cell[:, i] if busbar_orientation == "horizontal" else cell[i, :]
                for i in range(dim)]
    avg_profile = np.sum(profiles, axis=0)
    threshold   = np.percentile(avg_profile, 10)
    bb_idx      = np.where(avg_profile < threshold)[0]

    busbar_mask = np.zeros_like(cell, dtype=bool)
    if busbar_orientation == "horizontal":
        busbar_mask[bb_idx, :] = True
    else:
        busbar_mask[:, bb_idx] = True
    busbar_mask = ~busbar_mask  # True = NOT a busbar

    # percentile edge mask
    mask = cell > np.nanpercentile(cell, 1)

    # centre-fill to keep inner defects inside masked area
    center_mask = _center_square_mask(H, W, 0.9).astype(bool)
    center_mask = center_mask & (cell > 0)
    mask = mask | center_mask

    if not IBC:
        mask = mask & busbar_mask

    return mask


def validate_busbar_masks(cell_masks, cells_lo, cells_hi, nrows, ncols,
                          sample_idx=47):
    """
    Visualise busbar masks: module grid + one example overlap image.
    """
    masks_grid = [(m.astype(np.uint8) * 255) for m in cell_masks]
    plot_module_images(masks_grid, nrows=nrows, ncols=ncols, cmap="gray",
                       cbar=True, cbar_title="Mask (0/1)",
                       title="Busbar masks — all cells")

    i = sample_idx
    cell_img = cells_hi[i]
    mask     = cell_masks[i]
    overlap  = np.zeros_like(cell_img, dtype=np.float32)
    overlap[mask] = cell_img[mask]
    quick_plot(cell_masks[i].astype(np.uint8), title=f"Cell {i+1} mask", cmap="gray", cbar=True)
    quick_plot(cell_img,  title=f"Cell {i+1} hi-bias EL", cmap="inferno", cbar=True)
    quick_plot(overlap,   title=f"Cell {i+1} masked EL",  cmap="inferno", cbar=True)

In [ ]:

def load_module_data(mod_name, IBC, cell_dir, df):
    """
    Load and preprocess cell images + metadata for one module.

    Steps:
      1. Load images for high (80%) and low (20%) bias.
      2. Scale to 16-bit equivalent float32.
      3. Clip zero pixels inside the active mask to 1 (avoids log(0)).
      4. Normalise by sensor exposure time.
      5. Extract metadata row from the database.

    Returns dict with keys:
        cells_hi, cells_lo, cell_masks, mod_row,
        nrows, ncols, num_cells,
        lo_applied_I, lo_applied_V,
        hi_applied_I, hi_applied_V,
        hi_exposure_time, lo_exposure_time
    """
    mod_hi_path = os.path.join(cell_dir, f"{mod_name}_80")
    mod_lo_path = os.path.join(cell_dir, f"{mod_name}_20")
    assert os.path.isdir(mod_hi_path), f"Missing: {mod_hi_path}"
    assert os.path.isdir(mod_lo_path), f"Missing: {mod_lo_path}"

    # EDIT: original used cv2.IMREAD_GRAYSCALE (forces 8-bit) then rescaled by
    # 65535/255. Per your own notes elsewhere, your TIFFs are 16-bit and
    # should be loaded with IMREAD_UNCHANGED to preserve real precision.
    # Run the bit-depth check cell above first. If your TIFFs ARE 16-bit,
    # flip USE_UNCHANGED_READ to True below (and this function will skip the
    # redundant rescale, since the values are already in native 16-bit range).
    # Left as False by default to match Zubair's original behaviour until
    # you and your supervisor confirm the change.
    USE_UNCHANGED_READ = False  # EDIT: set True after confirming 16-bit source

    read_flag = cv2.IMREAD_UNCHANGED if USE_UNCHANGED_READ else cv2.IMREAD_GRAYSCALE

    cells_hi = [cv2.imread(os.path.join(mod_hi_path, f), read_flag)
                for f in sorted(os.listdir(mod_hi_path)) if f.endswith('.tiff')]
    cells_lo = [cv2.imread(os.path.join(mod_lo_path, f), read_flag)
                for f in sorted(os.listdir(mod_lo_path)) if f.endswith('.tiff')]

    num_cells = len(cells_hi)
    if   num_cells == 60: nrows, ncols = 6, 10   # landscape 10x6
    elif num_cells == 72: nrows, ncols = 6, 12   # portrait  12x6
    else:                 nrows, ncols = 6, num_cells // 6

    # extract database row
    parts   = mod_name.split("_")
    mod_ID  = int(parts[0]); mod_make = int(parts[1]); mod_model = int(parts[2])
    EL_date = datetime.strptime(parts[3], "%m%d%Y")
    EL_date = f"{EL_date.month}/{EL_date.day}/{EL_date.year}"
    mod_row = df[(df["Mod_ID"] == mod_ID) & (df["Make"] == mod_make) &
                 (df["Model"] == mod_model) & (df["High_EL_Date"] == EL_date)]
    assert len(mod_row) == 1, f"Metadata lookup failed for {mod_name}"

    # build busbar masks from the low-bias images
    cell_masks = [create_busbar_mask(c, IBC=IBC, busbar_orientation="horizontal")
                  for c in cells_lo]

    # 8-bit -> 16-bit float32 rescale (only meaningful if we actually read 8-bit)
    if not USE_UNCHANGED_READ:
        cells_hi = [c.astype(np.float32) * (65535.0 / 255.0) for c in cells_hi]
        cells_lo = [c.astype(np.float32) * (65535.0 / 255.0) for c in cells_lo]
    else:
        cells_hi = [c.astype(np.float32) for c in cells_hi]
        cells_lo = [c.astype(np.float32) for c in cells_lo]

    # zero -> 1 inside active mask (avoids log(0))
    cells_hi = [np.where(m & (c == 0), 1, c) for c, m in zip(cells_hi, cell_masks)]
    cells_lo = [np.where(m & (c == 0), 1, c) for c, m in zip(cells_lo, cell_masks)]

    # normalise by exposure time
    hi_exp = mod_row["High_Sensor_Exposure_Time_(s)"].values[0]
    lo_exp = mod_row["Low_Sensor_Exposure_Time_(s)"].values[0]
    cells_hi = [c / hi_exp for c in cells_hi]
    cells_lo = [c / lo_exp for c in cells_lo]

    return dict(
        cells_hi         = cells_hi,
        cells_lo         = cells_lo,
        cell_masks       = cell_masks,
        mod_row          = mod_row,
        nrows            = nrows,
        ncols            = ncols,
        num_cells        = num_cells,
        lo_applied_I     = mod_row["Low_Applied_Current_(A)"].values[0],
        lo_applied_V     = mod_row["Low_Applied_Voltage_(V)"].values[0],
        hi_applied_I     = mod_row["High_Applied_Current_(A)"].values[0],
        hi_applied_V     = mod_row["High_Applied_Voltage_(V)"].values[0],
        hi_exposure_time = hi_exp,
        lo_exposure_time = lo_exp,
    )


def validate_loaded_data(data):
    """Print exposure/applied conditions and display hi/lo module image grids."""
    d = data
    print(f"Hi-bias exposure: {d['hi_exposure_time']} s | "
          f"Lo-bias exposure: {d['lo_exposure_time']} s")
    print(f"Lo-bias:  I = {d['lo_applied_I']:.2f} A  V = {d['lo_applied_V']:.2f} V")
    print(f"Hi-bias:  I = {d['hi_applied_I']:.2f} A  V = {d['hi_applied_V']:.2f} V")
    plot_module_images(np.array(d['cells_hi']), nrows=d['nrows'], ncols=d['ncols'],
                       cmap="inferno", cbar=True, cbar_title="Pixel Intensity (a.u.)",
                       title="Hi-bias EL images")
    plot_module_images(np.array(d['cells_lo']), nrows=d['nrows'], ncols=d['ncols'],
                       cmap="inferno", cbar=True, cbar_title="Pixel Intensity (a.u.)",
                       title="Lo-bias EL images")

In [ ]:

def calculate_constant_f(cell_images, cell_masks, current_I, terminal_voltage_VT,
                          T_celsius, N=60):
    """
    Extract the constant factor f (Eq. 16).
    f accounts for the proportionality between EL intensity and local recombination current.

    Args:
        cell_images: list of 2-D float arrays (EL intensities, one per cell).
        cell_masks:  list of bool masks (True = active pixel).
        current_I:   applied current at low bias [A].
        terminal_voltage_VT: measured module terminal voltage at low bias [V].
        T_celsius:   cell temperature [degC].
        N:           total number of series cells.

    Returns:
        f (float)
    """
    U_th = k_B * (T_celsius + T_0) / q_e

    total = sum(
        (U_th / 2.0) * np.log(current_I / np.sum(1.0 / phi[mask]))
        for phi, mask in zip(cell_images, cell_masks)
    )

    f = np.exp((total - terminal_voltage_VT) / ((N / 2.0) * U_th))
    return f


def validate_constant_f(factor_f):
    """Print the extracted f value."""
    print(f"Extracted factor f = {factor_f:.4e}")

In [ ]:

def calculate_cell_voltage_Vi(phi_r, mask, current_I, factor_f, T_celsius):
    """
    Calculate low-bias terminal voltage Vi for a single cell (Eq. 13).

    Vi = (U_th/2) * ln( I / (f * integral(1/Phi(r) d2r)) )

    Returns:
        Vi (float) [V]
    """
    U_th = k_B * (T_celsius + T_0) / q_e
    integral = np.sum(1.0 / phi_r[mask])
    return (U_th / 2.0) * np.log(current_I / (factor_f * integral))


def validate_cell_voltage_Vi(Vi_values, lo_applied_V, cells_lo, nrows, ncols):
    """
    Check sum(Vi) ~= terminal voltage and plot Vi heatmap + lo-bias image grid.
    """
    sum_Vi = np.sum(Vi_values)
    print(f"Sum Vi = {sum_Vi:.3f} V  |  Measured V_T = {lo_applied_V:.3f} V  "
          f"|  Delta = {sum_Vi - lo_applied_V:.3f} V")

    Vi_arr = np.array(Vi_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(Vi_arr, annot=True, fmt=".3f", cmap="viridis",
                annot_kws={"size": 14}, cbar_kws={"label": "Vi (V)"}, ax=ax)
    ax.set_title("Low-bias cell voltage Vi")
    plt.show()

    plot_module_images(np.array(cells_lo), nrows=nrows, ncols=ncols,
                       cmap="inferno", cbar=True, cbar_title="Intensity (a.u.)",
                       title="Lo-bias EL (for comparison)")

In [ ]:

def calculate_calibration_constant_map(phi_r, Vi, mask, T_celsius):
    """
    Compute the pixel-wise calibration constant c(r) = Phi(r) . exp(-Vi/U_th)  (Eq. 8).

    Returns:
        c_r: float64 array, NaN outside active mask.
    """
    U_th = k_B * (T_celsius + T_0) / q_e
    c_r  = np.full_like(phi_r, np.nan, dtype=np.float64)
    c_r[mask] = phi_r[mask] / np.exp(Vi / U_th)
    return c_r


def validate_calibration_map(calibration_maps, cells_lo, cell_masks, Vi_values,
                              nrows, ncols):
    """
    Plot c(r) maps; scatter of per-cell mean c_i vs EL intensity and vs Vi.
    """
    plot_module_images(np.array(calibration_maps), nrows=nrows, ncols=ncols,
                       cmap="viridis", cbar=True, title="Calibration maps c(r)",
                       cbar_title="c(r) (a.u.)")

    c_i   = np.array([np.nanmean(c[m]) for c, m in zip(calibration_maps, cell_masks)])
    intens = [np.nanmedian(img[m]) for img, m in zip(cells_lo, cell_masks)]

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.scatter(c_i, intens, color='blue', label='Intensity')
    ax1.set_xlabel("Mean c_i per cell")
    ax1.set_ylabel("Median pixel intensity (a.u.)", color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax2 = ax1.twinx()
    ax2.scatter(c_i, Vi_values, color='orange', label='Vi')
    ax2.set_ylabel("Vi (V)", color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')
    fig.suptitle("c_i vs intensity and Vi")
    plt.show()

In [ ]:

def calculate_dark_saturation_current_map(mask, c_r, factor_f):
    """
    Compute pixel-wise J0(r) = f / c(r)  (Eq. 9).

    Returns:
        J0_map: masked float64 array (np.ma, NaN outside active mask).
    """
    J0 = np.full_like(c_r, np.nan, dtype=np.float64)
    J0[mask] = factor_f / c_r[mask]
    return np.ma.array(J0, mask=np.isnan(J0))


def validate_j0_maps(J0_maps, effective_J0_values, nrows, ncols):
    """
    Heatmap of per-cell effective J0, log10 maps across module, J0 histogram.
    """
    J0_arr = np.array(effective_J0_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(J0_arr, annot=True, fmt=".2e", cmap="viridis_r",
                annot_kws={"size": 14}, cbar_kws={"label": "Effective J0 (A)"}, ax=ax)
    ax.set_title("Effective J0 per cell")
    plt.show()

    cmap_j0 = plt.cm.viridis.copy()
    cmap_j0.set_bad(color='lightgray')
    plot_module_images(np.log10(np.array(J0_maps)), nrows=nrows, ncols=ncols,
                       cmap=cmap_j0, cbar=True, cbar_title="log10 J0 (A/px)",
                       title="log10 J0(r) maps")

    flat = np.log10(np.array(J0_maps).flatten())
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(flat[~np.isnan(flat)], bins=40, color='steelblue', alpha=0.8)
    ax.set_xlabel("log10 J0 (A/px)"); ax.set_ylabel("Frequency")
    ax.set_title("Distribution of J0 pixel values")
    plt.show()

    print(f"Module effective J0 = {np.sum(effective_J0_values):.2e} A")

In [ ]:

def calculate_high_bias_cell_voltage_Vi(phi_hi, c_r, T_celsius, mask):
    """
    Estimate high-bias terminal voltage Vi using the top 1% brightest active
    pixels as a proxy for the terminal region (Eq. 1).

    Assumption: negligible interconnect voltage drop -> slight underestimation
    of total module voltage.

    Returns:
        Vi_high (float) [V]
    """
    U_th      = k_B * (T_celsius + T_0) / q_e
    active_phi = phi_hi[mask]
    active_cr  = c_r[mask]

    p_hi = np.nanpercentile(active_phi, 99.9)
    p_lo = np.nanpercentile(active_phi, 98.9)
    idx  = np.where((active_phi <= p_hi) & (active_phi >= p_lo))

    Vi_candidates = U_th * np.log(active_phi[idx] / active_cr[idx])
    return float(np.nanmean(Vi_candidates))


def validate_high_bias_vi(Vi_hi_values, hi_applied_V, cells_hi, nrows, ncols):
    """
    Check sum(Vi_hi) ~= hi-bias terminal voltage and plot Vi heatmap + hi-bias images.
    """
    sum_Vi_hi = np.sum(Vi_hi_values)
    print(f"Sum Vi_hi = {sum_Vi_hi:.3f} V  |  Measured V_T = {hi_applied_V:.3f} V  "
          f"|  Delta = {sum_Vi_hi - hi_applied_V:.3f} V")

    Vi_hi_arr = np.array(Vi_hi_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(Vi_hi_arr, annot=True, fmt=".3f", cmap="viridis",
                annot_kws={"size": 18}, cbar_kws={"label": "Hi-bias Vi (V)"}, ax=ax)
    ax.set_title("High-bias cell voltage Vi")
    plt.show()

    plot_module_images(np.array(cells_hi), nrows=nrows, ncols=ncols,
                       cmap="inferno", cbar=True, cbar_title="Intensity (a.u.)",
                       title="Hi-bias EL (for comparison)")

In [ ]:

def calculate_local_voltage_map(phi_hi, c_r, T_celsius, mask):
    """
    Compute U(r) = U_th . ln(Phi_hi(r) / c(r))  (Eq. 1 rearranged).

    Returns:
        U_r: float64 array, NaN outside active mask.
    """
    U_th  = k_B * (T_celsius + T_0) / q_e
    U_r   = np.full_like(phi_hi, np.nan, dtype=np.float64)
    U_r[mask] = U_th * np.log(phi_hi[mask] / c_r[mask])
    return U_r


def validate_local_voltage_map(U_r_maps, cell_masks, hi_applied_V, nrows, ncols):
    """
    Plot U(r) maps; heatmap of per-cell mean U(r); compare sum(meanU) ~= V_T.
    """
    cmap_u = plt.cm.inferno.copy()
    cmap_u.set_bad(color='lightgray')
    plot_module_images(np.array(U_r_maps), nrows=nrows, ncols=ncols, cmap=cmap_u,
                       cbar=True, cbar_title="U(r) (V)", title="Local voltage maps U(r)")

    avg_U = [float(np.nanmean(U[m])) for U, m in zip(U_r_maps, cell_masks)]
    print(f"Sum meanU(r) = {np.sum(avg_U):.3f} V  |  Measured V_T = {hi_applied_V:.3f} V")

    arr = np.array(avg_U).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(arr, annot=True, fmt=".3f", cmap="inferno",
                annot_kws={"size": 14}, cbar_kws={"label": "Mean U(r) (V)"}, ax=ax)
    ax.set_title("Mean local voltage per cell")
    plt.show()

    return avg_U

In [ ]:

def calculate_local_current_density_map(J0_map, U_r_map, T_celsius, mask):
    """
    Compute J(r) = J0(r) . exp(U(r) / U_th)  (Eq. 2).

    Returns:
        J_r: float64 array, NaN outside active mask.
    """
    U_th = k_B * (T_celsius + T_0) / q_e
    J_r  = np.full_like(J0_map, np.nan, dtype=np.float64)
    J_r[mask] = J0_map[mask] * np.exp(U_r_map[mask] / U_th)
    return J_r


def validate_local_current_density_map(J_r_maps, cell_masks, hi_applied_I,
                                        nrows, ncols):
    """
    Plot J(r) maps; heatmap of integrated J per cell; compare median to I_applied.
    Returns list of integrated J values per cell.
    """
    cmap_j = plt.cm.viridis.copy()
    cmap_j.set_bad(color='lightgray')
    plot_module_images(np.array(J_r_maps), nrows=nrows, ncols=ncols, cmap=cmap_j,
                       cbar=True, cbar_title="J(r) (A/px)",
                       title="Local current density maps J(r)")

    int_J = [float(np.nansum(J[m])) for J, m in zip(J_r_maps, cell_masks)]

    arr = np.array(int_J).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(arr, annot=True, fmt=".2f", cmap="viridis",
                annot_kws={"size": 14}, cbar_kws={"label": "Integral J(r) (A)"}, ax=ax)
    ax.set_title("Integrated J(r) per cell")
    plt.show()

    print(f"Median Integral J(r) = {np.nanmedian(int_J):.3f} A  |  "
          f"Hi-bias applied I = {hi_applied_I:.3f} A")
    return int_J

In [ ]:

def calculate_series_resistance_map(Vi_high, U_r_map, J_r_map, mask):
    """
    Compute Rs(r) = (Vi - U(r)) / J(r)  (Eq. 4).

    Returns:
        Rs_r: float64 array, NaN outside active mask or where J(r) <= 0.
    """
    Rs_r = np.full_like(U_r_map, np.nan, dtype=np.float64)
    valid = mask & (J_r_map > 0)
    Rs_r[valid] = (Vi_high - U_r_map[valid]) / J_r_map[valid]
    return Rs_r


def calculate_cell_Rs_ohm(Vi_high, U_r_map, J_r_map, mask):
    """
    Compute current-weighted effective series resistance Rs,i for a cell [Ohm].

    Rs,i = Sum(dV . J(r)) / (Sum J(r))^2   where dV = |Vi - U(r)|
    """
    valid = mask & np.isfinite(U_r_map) & np.isfinite(J_r_map) & (J_r_map > 0)
    I_i  = np.nansum(J_r_map[valid])
    dV   = np.abs(Vi_high - U_r_map[valid])
    return float(np.nansum(dV * J_r_map[valid]) / I_i ** 2) if I_i > 0 else np.nan


def validate_series_resistance_map(Rs_r_maps, effective_Rs_i_values,
                                    Vi_hi_values, hi_applied_V, hi_applied_I,
                                    nrows, ncols):
    """
    Plot Rs(r) maps; heatmap of effective Rs,i; print module Rs and interconnect estimate.
    """
    cmap_rs = plt.cm.magma.copy()
    cmap_rs.set_bad(color='lightgray')
    plot_module_images(np.array(Rs_r_maps), nrows=nrows, ncols=ncols, cmap=cmap_rs,
                       cbar=True, cbar_title="Rs(r) (Ohm.px)",
                       title="Local series resistance maps Rs(r)")

    arr = np.array(effective_Rs_i_values).reshape(nrows, ncols)
    fig, ax = plt.subplots(figsize=(ncols * 2, nrows * 2))
    sns.heatmap(arr, annot=True, fmt=".5f", cmap="magma",
                annot_kws={"size": 14}, cbar_kws={"label": "Rs,i (Ohm)"}, ax=ax)
    ax.set_title("Effective Rs per cell")
    plt.show()

    Rs_cells     = np.nansum(effective_Rs_i_values)
    Rs_intercon  = (hi_applied_V - np.sum(Vi_hi_values)) / hi_applied_I
    total_Rs     = Rs_cells + Rs_intercon
    print(f"Cell Rs         = {Rs_cells:.4f} Ohm")
    print(f"Interconnect Rs = {Rs_intercon:.4f} Ohm  ({Rs_intercon/len(effective_Rs_i_values)*1e3:.3f} mOhm per cell)")
    print(f"Total module Rs = {total_Rs:.4f} Ohm")
    return total_Rs, Rs_intercon

In [ ]:

def extract_JscVoc(J, V):
    """Interpolate Isc (at V=0) and Voc (at I=0) from measured IV data."""
    Jsc = J[0] if not np.any(V < 0) else interp1d(
        V[_sign_idx(V) - 3 : _sign_idx(V) + 3],
        J[_sign_idx(V) - 3 : _sign_idx(V) + 3])(0).item()
    Voc = V[-1] if not np.any(J < 0) else interp1d(
        J[_sign_idx(J) - 3 : _sign_idx(J) + 3],
        V[_sign_idx(J) - 3 : _sign_idx(J) + 3])(0).item()
    return Jsc, Voc

def _sign_idx(arr):
    return np.where(np.diff(np.sign(arr)))[0][0]


def extract_IscVoc(I_curve, V_curve):
    """Interpolate Isc/Voc from a simulated (monotone) IV curve."""
    Isc = float(np.interp(0.0, V_curve, I_curve))
    sc  = np.where(np.diff(np.sign(I_curve)))[0]
    Voc = float(np.interp(0.0,
                          [I_curve[sc[0]], I_curve[sc[0] + 1]],
                          [V_curve[sc[0]], V_curve[sc[0] + 1]])) if len(sc) else np.nan
    return Isc, Voc


def extract_voc(I_curve, V_curve):
    """Return Voc from a simulated IV curve."""
    sc = np.where(np.diff(np.sign(I_curve)))[0]
    return float(np.interp(0.0,
                           [I_curve[sc[0]], I_curve[sc[0] + 1]],
                           [V_curve[sc[0]], V_curve[sc[0] + 1]])) if len(sc) else np.nan


def extract_pmpp(I_curve, V_curve):
    """Return (Pmpp, Vmp, Imp)."""
    P   = V_curve * I_curve
    idx = np.argmax(P)
    return P[idx], V_curve[idx], I_curve[idx]


def effective_module_j0(j0_cells):
    """Geometric mean of per-cell J0 values [A]."""
    j0 = np.asarray(j0_cells, dtype=float)
    return float(np.exp(np.mean(np.log(j0))))

In [ ]:

def fit_sdm_fixed_n(V, I, num_cells, T_celsius=25.0, n_fixed=1.0,
                    IL0=None, I00=None, Rs0=None, Rsh0=None,
                    Isc0=None, Voc0=None, method='lambertw'):
    """
    Fit a single-diode model with fixed ideality factor n.

    Fitted: IL [A], I0 [A], Rs [Ohm], Rsh [Ohm].
    Fixed:  nNsVth = n_fixed * num_cells * Vth.

    Isc0/Voc0: optional explicit values used for initial guesses when the
    IV array does not cleanly reach Isc or Voc (e.g. PySpice output).

    Returns dict with keys: IL, I0, Rs, Rsh, nNsVth, Vth, success, message, I_fit.
    """
    V = np.asarray(V, dtype=np.float64)
    I = np.asarray(I, dtype=np.float64)

    Vth         = k_B * (T_celsius + T_0) / q_e
    nNsVth_fix  = n_fixed * num_cells * Vth
    Voc_meas    = Voc0 if Voc0 is not None else float(V[-1])

    IL0  = max(Isc0 if Isc0 is not None else float(I[0]), 1e-6) if IL0 is None else IL0
    I00  = max(IL0 / (np.exp(Voc_meas / nNsVth_fix) - 1.0), 1e-15) if I00 is None else I00
    Rs0  = 0.2   if Rs0  is None else Rs0
    Rsh0 = 200.0 if Rsh0 is None else Rsh0

    x0 = np.array([np.log10(IL0), np.log10(I00), Rs0, np.log10(Rsh0)])
    lb = np.array([-6, -20,  0.0, 0.0])
    ub = np.array([ 3,  -1, 10.0, 8.0])

    def unpack(x):
        return 10**x[0], 10**x[1], x[2], 10**x[3]

    def residuals(x):
        IL, I0, Rs, Rsh = unpack(x)
        try:
            I_m = i_from_v(V, IL, I0, Rs, Rsh, nNsVth_fix, method=method)
        except Exception:
            return np.full_like(I, 1e6)
        res = I_m - I
        isc_pen = 5.0 * (I_m[np.argmin(np.abs(V))] - I[np.argmin(np.abs(V))])
        voc_pen = 5.0 * (I_m[np.argmin(np.abs(V - Voc_meas))]
                         - I[np.argmin(np.abs(V - Voc_meas))])
        return np.concatenate([res, [isc_pen, voc_pen]])

    fit = least_squares(residuals, x0, bounds=(lb, ub),
                        method='trf', loss='soft_l1', f_scale=0.05, max_nfev=5000)
    IL_f, I0_f, Rs_f, Rsh_f = unpack(fit.x)
    I_fit = i_from_v(V, IL_f, I0_f, Rs_f, Rsh_f, nNsVth_fix, method=method)

    return dict(IL=IL_f, I0=I0_f, Rs=Rs_f, Rsh=Rsh_f,
                nNsVth=nNsVth_fix, Vth=Vth, n_fixed=n_fixed,
                success=fit.success, message=fit.message, I_fit=I_fit)


def validate_iv_fit(fit_fixed_n, V, I, Isc, Voc, nNsVth, eff_I0,
                    eff_Rs_mod, Rsh, num_cells, n_fixed):
    """
    Overlay measured, diode-fit, and effective-module (PySpice SDM) IV curves.

    eff_I0 / eff_Rs_mod: effective module I0 [A] and Rs [Ohm] from PySpice SDM fit.
    """
    V_sim = np.linspace(0, Voc * 1.05, 300)

    I_diode = i_from_v(V_sim, fit_fixed_n['IL'], fit_fixed_n['I0'],
                        fit_fixed_n['Rs'], fit_fixed_n['Rsh'],
                        fit_fixed_n['nNsVth'], method='lambertw')
    I_eff   = i_from_v(V_sim, fit_fixed_n['IL'], eff_I0,
                        eff_Rs_mod, Rsh,
                        n_fixed * num_cells * k_B * (25 + T_0) / q_e,
                        method='lambertw')

    Voc_d, (Pmpp_d, _, _) = extract_voc(I_diode, V_sim), extract_pmpp(I_diode, V_sim)
    Voc_e, (Pmpp_e, _, _) = extract_voc(I_eff,   V_sim), extract_pmpp(I_eff,   V_sim)
    Pmpp_m, _, _           = extract_pmpp(I, V)

    print(f"\nDiode fit (n={n_fixed}):        I0={fit_fixed_n['I0']:.2e} A  "
          f"Rs={fit_fixed_n['Rs']:.4f} Ohm  Voc={Voc_d:.3f} V  Pmpp={Pmpp_d:.2f} W")
    print(f"Effective (PySpice SDM): I0={eff_I0:.2e} A  "
          f"Rs={eff_Rs_mod:.4f} Ohm  Voc={Voc_e:.3f} V  Pmpp={Pmpp_e:.2f} W")
    print(f"Measured:                Voc={Voc:.3f} V  Pmpp={Pmpp_m:.2f} W")

    fig, ax = plt.subplots(figsize=(20, 8))
    ax.plot(V, I, 'o', ms=4, label='Measured I-V', color='blue')
    ax.plot(V_sim, I_diode, '--', lw=4, label='Diode-fit I-V', color='green')
    ax.plot(V_sim, I_eff,   '-',  lw=2, label='Effective module (PySpice SDM)', color='red')
    ax.set_xlabel("Voltage (V)"); ax.set_ylabel("Current (A)")
    ax.set_title("Measured vs Diode-fit vs Effective module (PySpice SDM) I-V")
    ax.legend(loc='lower left')
    ax.set_xlim(0, Voc * 1.05); ax.set_ylim(0, Isc * 1.05)
    plt.subplots_adjust(right=0.50)

    row_labels = ["I0 (A)", "Rs (Ohm)", "Voc (V)", "Pmpp (W)"]
    cell_text  = [
        ["-", f"{fit_fixed_n['I0']:.2e}", f"{eff_I0:.2e}"],
        ["-", f"{fit_fixed_n['Rs']:.3f}", f"{eff_Rs_mod:.3f}"],
        [f"{Voc:.3f}",    f"{Voc_d:.2f}",   f"{Voc_e:.2f}"],
        [f"{Pmpp_m:.2f}", f"{Pmpp_d:.2f}",  f"{Pmpp_e:.2f}"],
    ]
    tbl = ax.table(cellText=cell_text, rowLabels=row_labels,
                   colLabels=["Measured", "Diode fit", "PySpice SDM"],
                   cellLoc='center', rowLoc='center', colLoc='center',
                   bbox=[0.157, 0.25, 0.42, 0.45])
    tbl.auto_set_font_size(False); tbl.set_fontsize(14)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:  cell.set_text_props(weight='bold'); cell.set_facecolor('#EAEAF2')
        if c == -1: cell.set_text_props(weight='bold'); cell.set_facecolor('#F5F5F5')
    plt.show()

    return V_sim, I_diode, I_eff

In [ ]:
# Requires: PySpice + ngspice (set up in the "NgSpice setup" cell above)

try:
    from PySpice.Spice.Netlist import Circuit, SubCircuit
    from PySpice.Unit import *
    from PySpice.Spice.NgSpice.Shared import NgSpiceShared
    _PYSPICE_AVAILABLE = True
except ImportError:
    _PYSPICE_AVAILABLE = False
    print("PySpice not found — skip PySpice cells.")


def build_pyspice_maps(eff_J0, eff_Rs, Rs_interconnect, fit_fn, num_cells, nrows, ncols):
    """
    Build per-cell PySpice parameter maps from Rajput-extracted values.

    The interconnect resistance is distributed equally across all cells as an
    additive offset to each cell's Rs:
        Rs_cell_i (PySpice) = Rs,i (Rajput) + Rs_interconnect / N

    Orientation: landscape (nrows x ncols) -> portrait (ncols x nrows) -> snake order.

    Returns dict with keys:
        rsmap, jmap, qmap, rshmap  — shape (portrait_rows, portrait_cols)
        portrait_rows, portrait_cols
        Rs_offset_per_cell  — the additive offset applied [Ohm]
    """
    portrait_rows = ncols
    portrait_cols = nrows

    Rs_offset = Rs_interconnect / num_cells
    eff_Rs_adj = [r + Rs_offset for r in eff_Rs]
    print(f"PySpice Rs offset per cell: {Rs_offset*1e3:.3f} mOhm  "
          f"(Rs_interconnect={Rs_interconnect:.4f} Ohm / {num_cells} cells)")

    def _to_map(values, fill_median=True):
        portrait = landscape_to_portrait_scalars(values, portrait_rows, portrait_cols)
        snake    = portrait_to_snake_order(portrait, portrait_rows, portrait_cols)
        arr      = np.array(snake, dtype=float).reshape(portrait_rows, portrait_cols)
        if fill_median:
            med = float(np.nanmedian(values))
            arr = np.where(np.isfinite(arr) & (arr > 0), arr, med)
        return arr

    rsmap  = _to_map(eff_Rs_adj)
    jmap   = _to_map(eff_J0)
    qmap   = np.full((portrait_rows, portrait_cols), fit_fn['IL'])
    rshmap = np.full((portrait_rows, portrait_cols), fit_fn['Rsh'] / num_cells)

    return dict(
        rsmap=rsmap, jmap=jmap, qmap=qmap, rshmap=rshmap,
        portrait_rows=portrait_rows, portrait_cols=portrait_cols,
        Rs_offset_per_cell=Rs_offset,
    )


def landscape_to_portrait_scalars(values, portrait_rows=10, portrait_cols=6):
    """
    Rotate a flat landscape-order scalar list to portrait row-major order.
    Landscape: portrait_cols rows x portrait_rows cols  (e.g. 6x10)
    Portrait:  portrait_rows rows x portrait_cols cols  (e.g. 10x6)
    """
    portrait = [None] * (portrait_rows * portrait_cols)
    for r_P in range(portrait_rows):
        for c_P in range(portrait_cols):
            landscape_idx = (portrait_cols - 1 - c_P) * portrait_rows + r_P
            portrait[r_P * portrait_cols + c_P] = values[landscape_idx]
    return portrait


def portrait_to_snake_order(cells, rows=10, cols=6):
    """
    Reorder portrait row-major cell list to PySpice snake-wiring order.
    Even columns: top->bottom; odd columns: bottom->top.
    """
    snake = [None] * (rows * cols)
    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            if c % 2 == 0:
                snake[idx] = cells[idx]
            else:
                snake[c * rows + (rows - 1 - r)] = cells[idx]
    return snake


def gen_cell_one_diode_rajput(name, q=1000.0 @ u_mA, rs=10 @ u_mOhm,
                               rsh=200 @ u_Ohm, j01=1e-12, ni1=1):
    """PySpice subcircuit for a single-diode solar cell."""
    cell = SubCircuit(name, 't_out', 't_in')
    cell.model('d1', 'D', IS=j01, N=ni1, RS=0)
    cell.I(1, 't_load', 't_in', q)
    cell.R(2, 't_load', 't_out', rs)
    cell.R(3, 't_in',   't_load', rsh)
    cell.Diode(4, 't_in', 't_load', model='d1')
    return cell


class BypassDiodeRajput(SubCircuit):
    __nodes__ = ('BPD_input', 'BPD_output')
    def __init__(self, name):
        SubCircuit.__init__(self, name, *self.__nodes__)
        self.model('BypassDiodeRajput', 'D',
                   IS=680e-12, RS=0.001, N=1.003,
                   CJO=1e-12, M=0.3, EG=0.69, XTI=6)
        self.Diode(1, 'BPD_input', 'BPD_output', model='BypassDiodeRajput')


def gen_module_oneD(rows=10, cols=6, rsmap=None, qmap=None, jmap=None,
                    rshmap=None, bypass=True):
    """Build a PySpice module Circuit (1-diode per cell, row-major series string)."""
    qmap   = np.ones((rows, cols)) * 10    if qmap   is None else qmap
    j1map  = np.ones((rows, cols)) * 1e-13 if jmap   is None else jmap
    rsmap  = np.ones((rows, cols)) * 0.01  if rsmap  is None else rsmap
    rshmap = np.ones((rows, cols)) * 5     if rshmap is None else rshmap

    ckt = Circuit('module')
    for r in range(rows):
        for c in range(cols):
            nm = f'cell_{r}_{c}'
            ckt.subcircuit(gen_cell_one_diode_rajput(
                nm,
                q   = float(qmap[r, c])   * 1.0 @ u_A,
                j01 = float(j1map[r, c]),
                rs  = float(rsmap[r, c])  * 1.0 @ u_Ohm,
                rsh = float(rshmap[r, c]) * 1.0 @ u_Ohm,
            ))
            ckt.X(nm, nm, r * cols + c + 1, r * cols + c + 2)

    if bypass:
        n = rows * cols
        pts = [(1, n // 3 + 1), (n // 3 + 1, 2 * n // 3 + 1), (2 * n // 3 + 1, n + 1)]
        for i, (a, b) in enumerate(pts):
            ckt.subcircuit(BypassDiodeRajput(f'bypass_r{i+1}'))
            ckt.X(f'bypass_r{i+1}', f'bypass_r{i+1}', a, b)

    ckt.V('input', ckt.gnd, 1, 0.0)
    ckt.R('meas', rows * cols + 1, 0, 0.001 @ u_Ohm)
    return ckt


def sim_module_oneD(rows=10, cols=6, rsmap=None, qmap=None, jmap=None,
                    rshmap=None, bypass=False, return_IV=False):
    """
    Simulate module I-V via PySpice DC sweep.

    Returns (mpp, vmp, imp, voc, isc) or with I, V appended if return_IV=True.
    """
    ckt      = gen_module_oneD(rows, cols, rsmap, qmap, jmap, rshmap, bypass)
    sim      = ckt.simulator(temperature=25, nominal_temperature=25,
                             simulator='ngspice-shared')
    analysis = sim.dc(Vinput=slice(-5, rows * cols * 0.9, 0.1))

    current = np.asarray(analysis.Vinput)
    voltage = np.asarray(analysis.sweep)
    if current.size == 0:
        empty = (0.0,) * 5
        return empty + (current, voltage) if return_IV else empty

    P       = current * voltage
    idx     = np.argmax(P)
    isc, voc = extract_JscVoc(J=current, V=voltage)
    result  = (P[idx], voltage[idx], current[idx], voc, isc)
    return result + (current, voltage) if return_IV else result


def validate_pyspice_sim(V, I, Isc, Voc, V_spice, I_spice, Pmpp_meas,
                          I_rajput_pvlib, V_sim, module_effective_J0,
                          total_Rs, effective_Rs_i_values):
    """Plot and tabulate PySpice vs pvlib vs measured IV curves."""
    mpp_spice = float(np.max(V_spice * I_spice))
    isc_s, voc_s = extract_JscVoc(I_spice, V_spice)
    _, imp_s, vmp_s = 0, 0, 0  # simplified

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot(V, I, 'o', ms=4, label='Measured', color='blue')
    ax.plot(V_sim, I_rajput_pvlib, '--', lw=2, label='Rajput (pvlib)', color='red')
    ax.plot(V_spice, I_spice, '-', lw=2, label='Rajput (PySpice)', color='darkorange')
    ax.set_xlabel("Voltage (V)"); ax.set_ylabel("Current (A)")
    ax.set_title("Measured vs Rajput I-V\n(pvlib module-level vs PySpice per-cell)")
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, Voc * 1.05); ax.set_ylim(0, Isc * 1.05)
    plt.tight_layout(); plt.show()

    print(f"PySpice: Isc={isc_s:.4f} A  Voc={voc_s:.4f} V  Pmpp={mpp_spice:.2f} W")
    print(f"Measured:          Isc={Isc:.4f} A  Voc={Voc:.4f} V  Pmpp={Pmpp_meas:.2f} W")

In [ ]:

def run_rajput_pipeline(mod_name, IBC, cell_dir, df, source_dir,
                         T_celsius=25, n_fixed=1,
                         plot=True, plot_final=True):
    """
    Run the full Rajput EL-to-IV extraction pipeline for one module.

    Steps:
        0. Load & preprocess images
        1. Factor f
        2. Low-bias cell voltage Vi
        3. Calibration map c(r)
        4. Dark saturation current J0(r)
        5. High-bias cell voltage Vi
        6. Local voltage U(r)
        7. Local current density J(r)
        8. Series resistance Rs(r)
        +  Sandia + constrained SDM IV fit

    Args:
        mod_name:    module identifier string (e.g. "2531_36_1_08062019")
        IBC:         True if IBC cell (suppresses busbar detection)
        cell_dir:    path to EL cell image directory
        df:          metadata DataFrame
        source_dir:  dataset root directory
        T_celsius:   measurement temperature [degC]
        n_fixed:     ideality factor for constrained SDM fit
        plot:        if True, produce all intermediate step diagnostic plots
        plot_final:  if True, produce the final measured vs diode-fit vs
                     PySpice SDM IV comparison plot

    Returns:
        dict with all intermediate and final results.
    """
    print(f"\n{'='*60}")
    print(f"  Module: {mod_name}  |  IBC={IBC}  |  T={T_celsius}degC")
    print(f"{'='*60}")

    # -- Step 0: load data ------------------------------------------------
    data      = load_module_data(mod_name, IBC, cell_dir, df)
    cells_hi  = data['cells_hi'];  cells_lo  = data['cells_lo']
    masks     = data['cell_masks']; mod_row   = data['mod_row']
    nrows     = data['nrows'];      ncols     = data['ncols']
    N         = data['num_cells']
    lo_I      = data['lo_applied_I']; lo_V = data['lo_applied_V']
    hi_I      = data['hi_applied_I']; hi_V = data['hi_applied_V']

    if plot:
        validate_loaded_data(data)
        validate_busbar_masks(masks, cells_lo, cells_hi, nrows, ncols)

    # -- Step 1: factor f -------------------------------------------------
    factor_f = calculate_constant_f(cells_lo, masks, lo_I, lo_V, T_celsius, N)
    if plot:
        validate_constant_f(factor_f)

    # -- Step 2: low-bias Vi ----------------------------------------------
    Vi_lo = [calculate_cell_voltage_Vi(phi, m, lo_I, factor_f, T_celsius)
             for phi, m in zip(cells_lo, masks)]
    if plot:
        validate_cell_voltage_Vi(Vi_lo, lo_V, cells_lo, nrows, ncols)

    # -- Step 3: calibration map c(r) -------------------------------------
    c_maps = [calculate_calibration_constant_map(phi, Vi, m, T_celsius)
              for phi, Vi, m in zip(cells_lo, Vi_lo, masks)]
    if plot:
        validate_calibration_map(c_maps, cells_lo, masks, Vi_lo, nrows, ncols)

    # -- Step 4: J0(r) maps -----------------------------------------------
    J0_maps    = [calculate_dark_saturation_current_map(m, c, factor_f)
                  for m, c in zip(masks, c_maps)]
    eff_J0     = [float(np.nansum(J[m])) for J, m in zip(J0_maps, masks)]
    if plot:
        validate_j0_maps(J0_maps, eff_J0, nrows, ncols)

    # -- Step 5: high-bias Vi ---------------------------------------------
    Vi_hi = [calculate_high_bias_cell_voltage_Vi(phi, c, T_celsius, m)
             for phi, c, m in zip(cells_hi, c_maps, masks)]
    if plot:
        validate_high_bias_vi(Vi_hi, hi_V, cells_hi, nrows, ncols)

    # -- Step 6: U(r) maps ------------------------------------------------
    U_maps = [calculate_local_voltage_map(phi, c, T_celsius, m)
              for phi, c, m in zip(cells_hi, c_maps, masks)]
    if plot:
        avg_U = validate_local_voltage_map(U_maps, masks, hi_V, nrows, ncols)
    else:
        avg_U = [float(np.nanmean(U[m])) for U, m in zip(U_maps, masks)]
    module_U = np.nansum(avg_U)

    # -- Step 7: J(r) maps ------------------------------------------------
    J_maps = [calculate_local_current_density_map(J0, U, T_celsius, m)
              for J0, U, m in zip(J0_maps, U_maps, masks)]
    if plot:
        int_J = validate_local_current_density_map(J_maps, masks, hi_I, nrows, ncols)
    else:
        int_J = [float(np.nansum(J[m])) for J, m in zip(J_maps, masks)]
    module_intJ = float(np.nanmedian(int_J))

    # -- Step 8: Rs(r) maps -----------------------------------------------
    Rs_maps = [calculate_series_resistance_map(Vi, U, J, m)
               for Vi, U, J, m in zip(Vi_hi, U_maps, J_maps, masks)]
    eff_Rs  = [calculate_cell_Rs_ohm(Vi, U, J, m)
               for Vi, U, J, m in zip(Vi_hi, U_maps, J_maps, masks)]
    if plot:
        total_Rs, Rs_interconnect = validate_series_resistance_map(
            Rs_maps, eff_Rs, Vi_hi, hi_V, hi_I, nrows, ncols)
    else:
        Rs_interconnect = (hi_V - np.sum(Vi_hi)) / hi_I
        total_Rs = np.nansum(eff_Rs) + Rs_interconnect

    # -- IV fit -----------------------------------------------------------
    IV_path = os.path.join(source_dir, "IV", mod_row["IVPath"].values[0])
    df_IV   = pd.read_csv(IV_path)
    V_meas  = df_IV["V"].values; I_meas = df_IV["I"].values

    Isc = mod_row['Isc_(A)'].values[0]; Voc = mod_row['Voc_(V)'].values[0]
    san = fit_sandia_simple(voltage=V_meas, current=I_meas, v_oc=Voc, i_sc=Isc)
    JL, J0_san, Rs_san, Rsh_san, nNsVth_san = san

    fit_fn = fit_sdm_fixed_n(V_meas, I_meas, N, T_celsius, n_fixed,
                              IL0=JL, I00=J0_san, Rs0=Rs_san, Rsh0=Rsh_san)
    
    # -- PySpice simulation -> SDM fit for effective module parameters -----
    spice_fit  = None
    spice_maps = None
    I_spice    = None
    V_spice    = None
    if _PYSPICE_AVAILABLE:
        spice_maps = build_pyspice_maps(
            eff_J0          = eff_J0,
            eff_Rs          = eff_Rs,
            Rs_interconnect = Rs_interconnect,
            fit_fn          = fit_fn,
            num_cells       = N,
            nrows           = nrows,
            ncols           = ncols,
        )
        (_, _, _, _, _,
         I_spice, V_spice) = sim_module_oneD(
            rows      = spice_maps['portrait_rows'],
            cols      = spice_maps['portrait_cols'],
            rsmap     = spice_maps['rsmap'],
            qmap      = spice_maps['qmap'],
            jmap      = spice_maps['jmap'],
            rshmap    = spice_maps['rshmap'],
            bypass    = False,
            return_IV = True,
        )
        # Fit SDM to PySpice I-V -> effective module parameters
        Isc_spice, Voc_spice = extract_IscVoc(I_spice, V_spice)
        spice_fit = fit_sdm_fixed_n(V_spice, I_spice, N, T_celsius, n_fixed,
                                     Isc0=Isc_spice, Voc0=Voc_spice)

    if (plot or plot_final) and spice_fit is not None:
        validate_iv_fit(
            fit_fn, V_meas, I_meas, Isc, Voc,
            nNsVth_san, spice_fit['I0'], spice_fit['Rs'], Rsh_san, N, n_fixed)
    elif plot or plot_final:
        print("PySpice unavailable — skipping effective module IV comparison.")

    print(f"\nSummary — {mod_name}")
    print(f"  factor f           = {factor_f:.4e}")
    print(f"  cell Rs (Rajput)   = {total_Rs:.4f} Ohm")
    if spice_fit is not None:
        print(f"  module I0 (PySpice SDM) = {spice_fit['I0']:.2e} A")
        print(f"  module Rs (PySpice SDM) = {spice_fit['Rs']:.4f} Ohm")
    print(f"  Isc                = {Isc:.4f} A")
    print(f"  Voc                = {Voc:.4f} V")

    return dict(
        mod_name=mod_name, data=data,
        factor_f=factor_f,
        Vi_lo=Vi_lo, c_maps=c_maps,
        J0_maps=J0_maps, eff_J0=eff_J0,
        Vi_hi=Vi_hi,
        U_maps=U_maps, avg_U=avg_U, module_U=module_U,
        J_maps=J_maps, int_J=int_J, module_intJ=module_intJ,
        Rs_maps=Rs_maps, eff_Rs=eff_Rs, total_Rs=total_Rs,
        Rs_interconnect=Rs_interconnect,
        Isc=Isc, Voc=Voc,
        JL=JL, J0_san=J0_san, Rs_san=Rs_san, Rsh_san=Rsh_san, nNsVth_san=nNsVth_san,
        fit_fn=fit_fn,
        V_meas=V_meas, I_meas=I_meas,
        spice_fit=spice_fit, spice_maps=spice_maps,
        I_spice=I_spice, V_spice=V_spice,
    )

## Run on your modules
EDIT: the module names below (`2531_36_1_08062019`, etc.) are Zubair's
example modules from his own subset of the dataset. Replace with your own
`module_code` values from `AnonDB_zub_60.csv` / your working CSV. Keep
`plot=True` only for the module(s) you want full diagnostic plots for —
it's slow; use `plot=False, plot_final=True` for a quick summary + final
IV comparison only.

In [ ]:

res1 = run_rajput_pipeline(
    mod_name   = "2531_36_1_08062019",  # EDIT: replace with your module code
    IBC        = True,
    cell_dir   = cell_dir,
    df         = df,
    source_dir = source_dir,
    T_celsius  = 25,
    n_fixed    = 1,
    plot       = True,
    plot_final = True,
)

In [ ]:
# Uncomment and extend once you've run more than one module through the pipeline.

# results  = [res1]  # EDIT: add res2, res3, ... as you run more modules
# names    = [r['mod_name'] for r in results]
# J0_vals  = [r['spice_fit']['I0'] if r['spice_fit'] else float('nan') for r in results]
# Rs_vals  = [r['spice_fit']['Rs'] if r['spice_fit'] else r['total_Rs'] for r in results]
# Isc_vals = [r['Isc'] for r in results]
# Voc_vals = [r['Voc'] for r in results]
#
# fig, axes = plt.subplots(1, 2, figsize=(16, 8))
# axes[0].bar(names, J0_vals, color='steelblue')
# axes[0].set_ylabel("Module I0 (A)"); axes[0].set_title("Effective I0 per module (PySpice SDM)")
# axes[0].set_xticklabels(names, rotation=30, ha='right')
#
# axes[1].bar(names, Rs_vals, color='darkorange')
# axes[1].set_ylabel("Module Rs (Ohm)"); axes[1].set_title("Effective Rs per module (PySpice SDM)")
# axes[1].set_xticklabels(names, rotation=30, ha='right')
#
# plt.tight_layout(); plt.show()
#
# print("\nSummary table:")
# print(f"{'Module':<30} {'I0 (A)':>12} {'Rs (Ohm)':>10} {'Isc (A)':>10} {'Voc (V)':>10}")
# for n, j, r, i, v in zip(names, J0_vals, Rs_vals, Isc_vals, Voc_vals):
#     print(f"{n:<30} {j:>12.2e} {r:>10.4f} {i:>10.4f} {v:>10.4f}")